In [5]:
import os
import re
from pathlib import Path

import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from transformers import BertTokenizerFast


# константы путей

DATA_DIR = Path("E:/PracticumMLMaterials/Yandex_Practicum")
SMALL_FILE = DATA_DIR / "raw_dataset_sample.csv"
FULL_FILE = DATA_DIR / "raw_dataset.csv"
TOKENIZED_FILE = DATA_DIR / "dataset_tokenized.pt"


# подготовка данных

def data_handling(min_len=7, max_len=28, use_smallfile=True, file_path=None):


    #Параметры:
     #   min_len     – минимальная длина текста в словах
     #   max_len     – максимальная длина текста в словах
     #   use_smallfile – использовать sample-файл (True) или полный (False)
     #   file_path   – если задан, то используется этот путь вместо SMALL/FULL

    #Возвращает:
     #   df          – DataFrame с очищенным текстом
     #   tokenizer   – BertTokenizerFast
     #   tokens      – BatchEncoding с tensорами input_ids и attention_mask


    if file_path is None:
        path = SMALL_FILE if use_smallfile else FULL_FILE
    else:
        path = Path(file_path)

    # читаем файл построчно
    with open(path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    df = pd.DataFrame({"text": lines})

    # ссылки → спец-слово 'link'
    df["text"] = df["text"].apply(
        lambda txt: re.sub(r"http\S+|www\.\S+", "link", txt)
    )

    # убираем лишние пробелы, приводим к нижнему регистру
    df["text"] = df["text"].str.strip().str.lower()

    # оставляем только латиницу, цифры и пробел
    df["text"] = df["text"].apply(
        lambda t: re.sub(r"[^a-z0-9\s]", "", t).strip()
    )

    # длина текста в словах
    df["text_length"] = df["text"].str.split().apply(len)

    # фильтр по минимальной длине
    df = df[df["text_length"] >= min_len].reset_index(drop=True)

    # обрезаем до max_len слов
    df["words"] = df["text"].str.split().apply(lambda x: x[:max_len])
    df["text"] = df["words"].str.join(" ")

    tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

    tokens = tokenizer(
        df["text"].tolist(),
        max_length=max_len,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    )

    return df, tokenizer, tokens


# dataset

class TextDataset(Dataset):


    def __init__(self, input_ids: torch.Tensor):
        # input_ids: (num_samples, seq_len)
        self.input_ids = input_ids

    def __len__(self):
        return self.input_ids.size(0)

    def __getitem__(self, idx):
        # возвращаем ровно одну последовательность токенов
        return self.input_ids[idx]   # (seq_len,)


# наша модель LSTM

class LSTMAutocompleteModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128, num_layers=1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
        )
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, input_ids, hidden=None):
        # input_ids: (batch, seq_len)
        x = self.embedding(input_ids)           # (batch, seq_len, embed_dim)
        out, hidden = self.lstm(x, hidden)      # (batch, seq_len, hidden_dim)
        logits = self.fc(out)                   # (batch, seq_len, vocab_size)
        return logits, hidden

    @torch.no_grad()
    def generate(self, start_seq, max_len=20, device="cpu"):
       
        #Автодополнение - берём start_seq и дописываем max_len токенов.
        #Возвращает список токенов: исходная последовательность + сгенерированные.
       
        self.eval()

        generated = start_seq.tolist()
        input_seq = start_seq.unsqueeze(0).to(device)  # (1, seq_len)
        hidden = None

        for _ in range(max_len):
            logits, hidden = self.forward(input_seq, hidden)  # logits: (1, cur_len, vocab)
            next_token = torch.argmax(logits[:, -1, :], dim=-1)  # (1,)
            generated.append(next_token.item())

            # расширяем входную последовательность предсказанным токеном
            input_seq = torch.cat([input_seq, next_token.unsqueeze(0)], dim=1)

        return generated


# Задаём Rouge

def rouge_l(ref_tokens, pred_tokens):

    if len(ref_tokens) == 0 or len(pred_tokens) == 0:
        return 0.0

    m, n = len(ref_tokens), len(pred_tokens)
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    # классический LCS
    for i in range(m):
        for j in range(n):
            if ref_tokens[i] == pred_tokens[j]:
                dp[i + 1][j + 1] = dp[i][j] + 1
            else:
                dp[i + 1][j + 1] = max(dp[i][j + 1], dp[i + 1][j])

    lcs = dp[m][n]

    recall = lcs / m
    precision = lcs / n

    if recall + precision == 0:
        return 0.0

    return 2 * recall * precision / (recall + precision)


# тренировка с выводом loss + ROUGE-L

def train_model_with_rouge(
    model,
    dataloader,
    pad_token_id,
    epochs=3,
    lr=1e-3,
    device="cuda",
    rouge_samples_per_batch=4,
):
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0

        total_rouge = 0.0
        total_rouge_count = 0

        pbar = tqdm(dataloader, desc=f"Epoch {epoch}/{epochs}")
        # Считаем средний лосс по числу обработанных батчей
        for batch_idx, batch_seqs in enumerate(pbar, start=1):
            # batch_seqs: (batch, seq_len)
            batch_seqs = batch_seqs.to(device)

            # подготовка X, Y 
            X = batch_seqs[:, :-1]    
            Y = batch_seqs[:, 1:]      

            optimizer.zero_grad()
            logits, _ = model(X)       # logits: (batch, seq_len-1, vocab_size)

            loss = criterion(
                logits.reshape(-1, logits.size(-1)),   # (batch*(seq_len-1), vocab_size)
                Y.reshape(-1),                         # (batch*(seq_len-1))
            )
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

            #  ROUGE-L по автодополнению
            with torch.no_grad():
                batch_size = batch_seqs.size(0)
                n_examples = min(rouge_samples_per_batch, batch_size)

                for i in range(n_examples):
                    seq = batch_seqs[i]  # (seq_len,)
                    # убираем pad токены
                    seq_ids = [t for t in seq.tolist() if t != pad_token_id]
                    if len(seq_ids) < 4:
                        continue

                    # делим: 3/4 — вход, 1/4 — таргет
                    split = int(len(seq_ids) * 0.75)
                    if not (0 < split < len(seq_ids)):
                        continue

                    prompt_ids = seq_ids[:split]
                    target_ids = seq_ids[split:]

                    start_seq = torch.tensor(prompt_ids, dtype=torch.long, device=device)
                    gen_full = model.generate(
                        start_seq,
                        max_len=len(target_ids),
                        device=device,
                    )
                    # берём только дополнение
                    pred_completion = gen_full[len(prompt_ids):]

                    score = rouge_l(target_ids, pred_completion)
                    total_rouge += score
                    total_rouge_count += 1

            # средний лосс по числу уже пройденных батчей
            avg_loss_so_far = total_loss / batch_idx
            avg_rouge_so_far = (
                total_rouge / total_rouge_count if total_rouge_count > 0 else 0.0
            )
            pbar.set_postfix(loss=avg_loss_so_far, rouge_l=avg_rouge_so_far)

        avg_loss = total_loss / len(dataloader)
        avg_rouge = total_rouge / total_rouge_count if total_rouge_count > 0 else 0.0
        print(f"\nEpoch {epoch}: loss={avg_loss:.4f}, ROUGE-L={avg_rouge:.4f}\n")

    return model


# Body

def main():
    # загружаем данные и токенизируем
    df, tokenizer, tokens = data_handling(
        min_len=7,
        max_len=28,
        use_smallfile=True,

    )

    input_ids = tokens["input_ids"] 
    pad_token_id = tokenizer.pad_token_id

    # Dataset + DataLoader
    dataset = TextDataset(input_ids)
    dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

    # Модель
    vocab_size = tokenizer.vocab_size
    model = LSTMAutocompleteModel(vocab_size)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Обучение с выводом loss и ROUGE-L
    model = train_model_with_rouge(
        model,
        dataloader,
        pad_token_id=pad_token_id,
        epochs=3,
        lr=1e-3,
        device=device,
        rouge_samples_per_batch=4,  # можно увеличить/уменьшить
    )

    # Сохраняем модель
    torch.save(model.state_dict(), DATA_DIR / "lstm_autocomplete_with_rouge.pt")


if __name__ == "__main__":
    main()


Epoch 1/3: 100%|██████████| 9728/9728 [41:49<00:00,  3.88it/s, loss=4.04, rouge_l=0.195] 



Epoch 1: loss=4.0359, ROUGE-L=0.1950



Epoch 2/3: 100%|██████████| 9728/9728 [41:55<00:00,  3.87it/s, loss=3.69, rouge_l=0.21] 



Epoch 2: loss=3.6852, ROUGE-L=0.2100



Epoch 3/3: 100%|██████████| 9728/9728 [41:53<00:00,  3.87it/s, loss=3.58, rouge_l=0.211]



Epoch 3: loss=3.5835, ROUGE-L=0.2105

